### Neural Network from Scratch


In [53]:
from abc import abstractmethod

In [ ]:
class Layer:
    def __init__(self, input, output, input_shape, output_shape):

        self.input = input
        self.output = output
        self.input_shape = input_shape
        self.output_shape = output_shape

    @abstractmethod
    def output(self):
        return self.output

    @abstractmethod
    def input(self):
        return self.input

    @abstractmethod
    def input_shape(self):
        return self.input_shape

    @abstractmethod
    def output_shape(self):
        return self.output_shape

    @abstractmethod
    def forward_propagation(self):
        pass

    @abstractmethod
    def backward_propagation(self):
        pass

### Tạo lớp Fully connected layer


In [55]:
import numpy as np

In [ ]:
class FCLayer(Layer):
    def __init__(self, input_shape, output_shape):
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.weights = np.random.rand(input_shape, output_shape) - 0.5
        self.bias = np.random.rand(1, output_shape) - 0.5

    def forward_propagation(self, input):
        self.input = input
        self.output = np.dot(self.input, self.weights) + self.bias
        return self.output

    def backward_propagation(self, output_error, learning_rate):
        # Vì sao ở đây không nhân đạo hàm hàm kích hoạt?
        # Lý do là vì hàm kích hoạt đã được tính trong hàm forward_propagation của activation layer trước khi
        # truyền ngược lại cho layer này
        # input_error: đạo hàm của hàm mất mát của đầu ra layer hiện tại (l-1) (gs layer tiếp theo là l)
        input_error = np.dot(output_error, self.weights.T)

        # weights_error: đạo hàm của hàm mất mát theo weights
        # => Công thức: weights_error = (đao hàm của hàm mất mát theo z của layer hiện tại) * (output của layer trước đó)
        # Mà đạo hàm của hàm mất mát theo z của layer hiện tại chính là output_error (𝛿)
        weights_error = np.dot(self.input.T, output_error)
        self.weights -= learning_rate * weights_error
        self.bias -= learning_rate * output_error
        return input_error

In [ ]:
class ActivationLayer(Layer):
    def __init__(self, input_shape, output_shape, activation, activation_prime):
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.activation = activation
        self.activation_prime = activation_prime

    def forward_propagation(self, input):
        # input: đầu vào (z) của activation layer, ví dụ có công thức là sigmoid(input) = 1/(1+e^(-input))
        self.input = input
        self.output = self.activation(self.input)
        return self.output

    # output_error là 𝛿 (Gradient của hàm lỗi đối với đầu ra của layer l) 
    def backward_propagation(self, output_error, learning_rate):
        return self.activation_prime(self.input) * output_error

In [ ]:
class Network:
    def __init__(self):
        self.layers = []
        self.loss = None
        self.loss_prime = None

    def add(self, layer):
        self.layers.append(layer)

    def setup_loss(self, loss, loss_prime):
        self.loss = loss
        self.loss_prime = loss_prime

    def predict(self, input):
        result = []
        n = len(input)

        # Dự đoán cho từng mẫu dữ liệu
        for i in range(n):
            output = input[i]
            for layer in self.layers:
                output = layer.forward_propagation(output)
            result.append(output)
        return result

    def fit(self, X_train, y_train, epochs, learning_rate):
        n = len(X_train)

        # Huấn luyện cho từng epoch
        for i in range(epochs):
            err = 0
            for j in range(n):
                output = X_train[j]

                for layer in self.layers:
                    output = layer.forward_propagation(output)

                err += self.loss(y_train[j], output)

                # Lan truyền ngược
                error = self.loss_prime(y_train[j], output)

                for layer in reversed(self.layers):
                    error = layer.backward_propagation(error, learning_rate)

            err /= n
            print("epoch %d/%d   error=%f" % (i + 1, epochs, err))

### Xây dựng kiến trúc mạng


In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return (z > 0).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z) * (1 - sigmoid(z))

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_prime(z):
    return (z > 0).astype(int)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_prime(z):
    return sigmoid(z) * (1 - sigmoid(z))

In [ ]:
def loss(y_true, y_pred):
    return np.mean(np.power(y_true - y_pred, 2))


def loss_prime(y_true, y_pred):
    return 2 * (y_pred - y_true) / np.size(y_true)

In [ ]:
X_train = np.array([[[0, 0]], [[0, 1]], [[1, 0]], [[1, 1]]])
y_train = np.array([[[0]], [[1]], [[1]], [[0]]])

In [ ]:
# input, output là theo data, còn ở hidden layer có bao nhiêu neural là tuỳ mùnh

network = Network()

# FC Layer 1: Chuyển từ input (2 neurons) → hidden layer (4 neurons).
network.add(FCLayer(2, 4))  # network.add(FCLayer((1,2),(1,3|4|5...))) cũng được
network.add(
    ActivationLayer((1, 4), (1, 4), relu, relu_prime)
)  # network.add(FCLayer((1,5),(1,5))) cũng được nếu ở trên chọn 5 neural ở hidden layer đầu

# FC Layer 2: Chuyển từ hidden layer (4 neurons) → output layer (1 neuron).
network.add(FCLayer(4, 1))
network.add(ActivationLayer((1, 1), (1, 1), sigmoid, sigmoid_prime))

In [63]:
network.setup_loss(loss, loss_prime)
network.fit(X_train, y_train, epochs=1000, learning_rate=0.01)

epoch 1/1000   error=0.257666
epoch 2/1000   error=0.257525
epoch 3/1000   error=0.257386
epoch 4/1000   error=0.257249
epoch 5/1000   error=0.257112
epoch 6/1000   error=0.256977
epoch 7/1000   error=0.256843
epoch 8/1000   error=0.256710
epoch 9/1000   error=0.256579
epoch 10/1000   error=0.256448
epoch 11/1000   error=0.256319
epoch 12/1000   error=0.256191
epoch 13/1000   error=0.256064
epoch 14/1000   error=0.255938
epoch 15/1000   error=0.255813
epoch 16/1000   error=0.255689
epoch 17/1000   error=0.255567
epoch 18/1000   error=0.255445
epoch 19/1000   error=0.255324
epoch 20/1000   error=0.255205
epoch 21/1000   error=0.255086
epoch 22/1000   error=0.254969
epoch 23/1000   error=0.254852
epoch 24/1000   error=0.254737
epoch 25/1000   error=0.254622
epoch 26/1000   error=0.254508
epoch 27/1000   error=0.254395
epoch 28/1000   error=0.254283
epoch 29/1000   error=0.254172
epoch 30/1000   error=0.254062
epoch 31/1000   error=0.253953
epoch 32/1000   error=0.253844
epoch 33/1000   e

In [ ]:
def mapping_output(output):
    if output >= 0.5:
        return 1
    else:
        return 0

In [ ]:
X_test = np.array([[0, 1]])
output = network.predict(X_test)

mapping_output(output[0])

1